# Analisis Framing Bisnis Olist Dataset
**Framing 1: Masalah utama adalah keterlambatan pengiriman (delivery delay).**

Notebook ini berisi langkah-langkah untuk memproses dataset publik E-Commerce Olist untuk menetapkan kerangka keputusan bisnis berdasarkan instruksi tugas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Memuat Dataset
Pastikan dataset `.csv` dari Kaggle berada di folder yang sama dengan notebook ini.

In [ ]:
try:
    orders = pd.read_csv('olist_orders_dataset.csv')
    items = pd.read_csv('olist_order_items_dataset.csv')
    payments = pd.read_csv('olist_order_payments_dataset.csv')
    reviews = pd.read_csv('olist_order_reviews_dataset.csv')
    print("Semua dataset berhasil dimuat.")
except FileNotFoundError as e:
    print("Dataset tidak lengkap atau tidak ditemukan. Silakan ekstrak file csv di folder ini.\nError:", e)

## 2. Menggabungkan Dataset (Level-Order)
Sesuai regulasi tugas, kita perlu membangun dataset level-order melalui penggabungan tabel yang relevan.

In [ ]:
# Agregasi data items, payments, dan reviews per order_id
order_items = items.groupby('order_id').agg({'price': 'sum', 'freight_value': 'sum'}).reset_index()
order_payments = payments.groupby('order_id').agg({'payment_value': 'sum'}).reset_index()
order_reviews = reviews.groupby('order_id').agg({'review_score': 'mean'}).reset_index()

# Proses join (menggabungkan tabel-tabel relevan)
df = orders.merge(order_items, on='order_id', how='left')
df = df.merge(order_payments, on='order_id', how='left')
df = df.merge(order_reviews, on='order_id', how='left')

# Konversi string kolom tanggal menjadi format Datetime
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Fokus hanya pada pesanan yang sudah memiliki tanggal diterima (delivered)
df = df.dropna(subset=['order_delivered_customer_date'])
df.head()

## 3. Membuat Variabel Turunan
Membuat dua variabel sesuai dengan Framing 1 (Delivery) yang dipilih:
- `delivery_duration` (durasi pengiriman)
- `timeliness_status` (on-time vs late)

In [ ]:
# Variabel 1: Delivery Duration (Durasi Pengiriman aktual dari pembelian hingga pelanggan terima)
df['delivery_duration'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# Variabel 2: Timeliness Status (Ketepatan Waktu: On-Time vs Late)
df['timeliness_status'] = np.where(df['order_delivered_customer_date'] > df['order_estimated_delivery_date'], 'Late', 'On-Time')

df[['order_id', 'delivery_duration', 'timeliness_status']].head()

## 4. Analisis Eksploratif Dasar
Perbandingan rata-rata untuk menemukan justifikasi *decision framing*.

In [ ]:
print("=== TINGKAT KETEPATAN PENGIRIMAN GLOBAL ===")
status_counts = df['timeliness_status'].value_counts(normalize=True) * 100
display(pd.DataFrame(status_counts).rename(columns={'proportion': 'Persentase (%)', 'timeliness_status': 'Persentase (%)'}).round(2))

print("\n=== RATA-RATA SKOR ULASAN (REVIEW SCORE) ===")
avg_score = df.groupby('timeliness_status')[['review_score']].mean()
display(avg_score.round(2))

print("\n=== PROPORSI ULASAN BINTANG 1 ===")
# Identifikasi agregat ulasan dengan mutlak Bintang 1 (Sangat Buruk), lalu ubah ke int agar kalkulasi rerata berhasil
df['is_1_star'] = (df['review_score'] <= 1.0).astype(int)
bintang1_prop = (df.groupby('timeliness_status')[['is_1_star']].mean() * 100)
bintang1_prop.columns = ['Proporsi Bintang 1 (%)']
display(bintang1_prop.round(2))

## 5. Visualisasi Baseline
Grafik *Late Delivery Rate* (tingkat pesanan terlambat pertengahan tahun).

In [ ]:
# Ekstrak bulan pemesanan untuk melihat tren waktu
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')

# Hitung persentase pesanan terlambat setiap bulannya
trend = df.groupby('purchase_month')['timeliness_status'].apply(lambda x: (x == 'Late').mean() * 100)

plt.figure(figsize=(12, 6))
trend.plot(kind='bar', color='coral', edgecolor='black')

plt.title('Tingkat Keterlambatan Pengiriman Bulanan (Late Delivery Rate) - Olist', fontsize=14)
plt.xlabel('Bulan Pemesanan', fontsize=12)
plt.ylabel('Persentase Pesanan Terlambat (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()